In [1]:
# !python3 ../scripts/download.py --help

In [2]:
!uv pip install plotly nbformat scikit-learn

Using Python 3.12.5 environment at: C:\Users\barrt\PycharmProjects\disinformation\.venv
Audited 3 packages in 9ms


In [3]:
# !uv run ../scripts/download.py --subreddit aliens --n-posts 500 --sort controversial --replace-more-limit 0

In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [5]:
df = pd.read_csv("../data/data_dump202601082211.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

if df["edited"].dtype == "object":
    df["edited"] = df["edited"].map({True: True, False: False, "True": True, "False": False})

print(f"Total rows: {len(df)}")
print(f"Posts: {len(df[df['activity_type'] == 'post'])}")
print(f"Comments: {len(df[df['activity_type'] == 'comment'])}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")

df.head(20)

Total rows: 34030
Posts: 381
Comments: 33649

Columns: ['activity_id', 'activity_type', 'timestamp', 'subreddit', 'author', 'parent_id', 'parent_type', 'content', 'permalink', 'score', 'upvotes', 'downvotes', 'upvote_ratio', 'num_comments', 'edited']

Date range: 2014-01-26 21:00:24+00:00 to 2026-01-08 16:03:03+00:00


,activity_id,activity_type,timestamp,subreddit,author,parent_id,parent_type,content,permalink,score,upvotes,downvotes,upvote_ratio,num_comments,edited
0,t3_16i88dt,post,2023-09-14 04:03:43+00:00,aliens,imaginexus,NaN,NaN,A good summary from X on the alien mummy situa...,https://www.reddit.com/r/aliens/comments/16i88...,2063,2063,0.0,0.56,2764.0,False
1,t1_k0i921c,comment,2023-09-14 04:03:44+00:00,aliens,AutoModerator,t3_16i88dt,post,__Reminder__: Read the rules and understand th...,https://www.reddit.com/r/aliens/comments/16i88...,1,1,NaN,NaN,NaN,False
2,t1_k0irteh,comment,2023-09-14 07:28:15+00:00,aliens,Jasonclark2,t3_16i88dt,post,![gif](giphy|7ziXfljX6nkpGPndlm),https://www.reddit.com/r/aliens/comments/16i88...,243,243,NaN,NaN,NaN,False
3,t1_k0j3ghu,comment,2023-09-14 10:02:13+00:00,aliens,angomania,t3_16i88dt,post,"""UPDATE: I've investigated further, and I've d...",https://www.reddit.com/r/aliens/comments/16i88...,1174,1174,NaN,NaN,NaN,False
4,t1_k0ja5eq,comment,2023-09-14 11:16:27+00:00,aliens,NaN,t3_16i88dt,post,The coping in these comments is real,https://www.reddit.com/r/aliens/comments/16i88...,80,80,NaN,NaN,NaN,False
5,t1_k0jgys0,comment,2023-09-14 12:16:51+00:00,aliens,Strudol,t3_16i88dt,post,"If these were any kind of real, they wouldn’t ...",https://www.reddit.com/r/aliens/comments/16i88...,466,466,NaN,NaN,NaN,True
6,t1_k0id4ie,comment,2023-09-14 04:42:09+00:00,aliens,Kat_Konstanze,t3_16i88dt,post,> no radius/ulna; tibula/fibula\n\nSo... they ...,https://www.reddit.com/r/aliens/comments/16i88...,811,811,NaN,NaN,NaN,False
7,t1_k0jifda,comment,2023-09-14 12:28:35+00:00,aliens,robtbo,t3_16i88dt,post,These ‘professionals’ are about to have their ...,https://www.reddit.com/r/aliens/comments/16i88...,84,84,NaN,NaN,NaN,False
8,t1_k0ixxq1,comment,2023-09-14 08:49:56+00:00,aliens,darthbeefwellington,t3_16i88dt,post,For anyone interested in the NCBI datasets (DN...,https://www.reddit.com/r/aliens/comments/16i88...,141,141,NaN,NaN,NaN,False
9,t1_k0jaic1,comment,2023-09-14 11:19:58+00:00,aliens,NaN,t3_16i88dt,post,This screams fake to me. Could be part of the ...,https://www.reddit.com/r/aliens/comments/16i88...,45,45,NaN,NaN,NaN,False


In [6]:
# Ile unikalnych użytkowników?
print(f"Unique users: {df['author'].nunique()}")
print(f"Deleted/removed authors: {(df['author'] == '').sum()}")

# Aktywność użytkowników - top 10
print("\n=== TOP 10 most active users ===")
top_users = df['author'].value_counts().head(10)
print(top_users)

# Wykres top users
fig = px.bar(x=top_users.values, y=top_users.index, orientation='h',
             title='Top 10 Most Active Users',
             labels={'x': 'Number of Posts/Comments', 'y': 'User'},
             color=top_users.values,
             color_continuous_scale='Viridis')
fig.update_layout(height=400, showlegend=False)
fig.show()

Unique users: 14045
Deleted/removed authors: 0

=== TOP 10 most active users ===
author
AutoModerator        321
aliens-ModTeam       222
lickem369             91
Funkadelick99         75
AntisocialGuru        64
MilkTeaPetty          59
Seekertwentyfifty     52
Alien_Element         45
Pleasant-Lie-9053     45
Mn4by                 45
Name: count, dtype: int64


In [7]:
# Rozkład aktywności czasowej
df['hour'] = df['timestamp'].dt.hour
df['date'] = df['timestamp'].dt.date

print("=== Posts per day ===")
posts_per_day = df[df['activity_type'] == 'post']['date'].value_counts().sort_index()
print(posts_per_day)

print("\n=== Comments per day ===")
comments_per_day = df[df['activity_type'] == 'comment']['date'].value_counts().sort_index()
print(comments_per_day)

print("\n=== Activity by hour ===")
activity_by_hour = df['hour'].value_counts().sort_index()
print(activity_by_hour)

# Wykresy czasowe
fig = make_subplots(rows=2, cols=1, subplot_titles=('Activity by Date', 'Activity by Hour'),
                     specs=[[{"secondary_y": False}], [{"secondary_y": False}]])

# Posts & Comments per day
fig.add_trace(go.Scatter(x=posts_per_day.index, y=posts_per_day.values, 
                         name='Posts', mode='lines+markers', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=comments_per_day.index, y=comments_per_day.values, 
                         name='Comments', mode='lines+markers', line=dict(color='orange')), row=1, col=1)

# Activity by hour
fig.add_trace(go.Bar(x=activity_by_hour.index, y=activity_by_hour.values, 
                     name='Activity', marker=dict(color='green')), row=2, col=1)

fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Hour of Day", row=2, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_layout(height=600, hovermode='x unified')
fig.show()

=== Posts per day ===
date
2014-01-26    1
2015-10-28    1
2018-02-19    1
2018-09-09    1
2019-03-09    1
             ..
2025-12-17    1
2025-12-28    1
2025-12-29    1
2026-01-06    1
2026-01-07    1
Name: count, Length: 318, dtype: int64

=== Comments per day ===
date
2014-01-26     3
2014-01-27     6
2014-01-28     5
2015-10-28    11
2015-10-30     2
              ..
2026-01-01     1
2026-01-02     1
2026-01-06    50
2026-01-07    89
2026-01-08    52
Name: count, Length: 864, dtype: int64

=== Activity by hour ===
hour
0     1696
1     1636
2     1450
3     1777
4     1693
5     1319
6     1168
7     1002
8      830
9      770
10     983
11    1046
12    1177
13    1420
14    1518
15    1503
16    1541
17    1726
18    1590
19    1671
20    1525
21    1565
22    1722
23    1702
Name: count, dtype: int64


In [8]:
# Statystyki score (upvotes/downvotes)
print("=== Score statistics ===")
print(f"Avg score: {df['score'].mean():.2f}")
print(f"Median score: {df['score'].median():.2f}")
print(f"Max score: {df['score'].max()}")
print(f"Min score: {df['score'].min()}")

print("\n=== Top 10 highest scored posts/comments ===")
top_scores = df.nlargest(10, 'score')[['author', 'activity_type', 'score', 'content']]
for idx, row in top_scores.iterrows():
    print(f"{row['author']} ({row['activity_type']}): {row['score']} | {row['content'][:60]}...")

# Wykresy score
fig = make_subplots(rows=1, cols=2, subplot_titles=('Score Distribution', 'Top 10 Highest Scores'))

# Histogram rozkładu score
fig.add_trace(go.Histogram(x=df['score'], nbinsx=30, name='Score Distribution',
                           marker=dict(color='lightblue')), row=1, col=1)

# Top 10 scores
top_10_scores = df.nlargest(10, 'score')[['author', 'score']]
fig.add_trace(go.Bar(x=top_10_scores['author'], y=top_10_scores['score'],
                     name='Top Scores', marker=dict(color='crimson')), row=1, col=2)

fig.update_xaxes(title_text="Score", row=1, col=1)
fig.update_xaxes(title_text="Author", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_yaxes(title_text="Score", row=1, col=2)
fig.update_layout(height=400, showlegend=False)
fig.show()

=== Score statistics ===
Avg score: 11.96
Median score: 2.00
Max score: 45507
Min score: -143

=== Top 10 highest scored posts/comments ===
GoldIsAMetal (post): 45507 | More Photos from Mexico UFO Hearings

These images were from...
Streay (post): 13835 | Tomb Raiders alleged photos in the Nazca Caves

URL: https:/...
nan (post): 11741 | Nothing to see in the bottom left corner here.

URL: https:/...
nan (post): 11131 | Leaked footage of grave robbers raiding Nazca Cave in Peru e...
imaginexus (post): 8160 | r/aliens finally gets its alien: The University of Ica just ...
ComonomoC (comment): 4114 | Man, digital photographs have gotten so much better in 5 yea...
Baggizine (comment): 3719 | Professional Moon Enjoyer here, this is fake.

The angle of ...
WesterlyStraight (comment): 3433 | Translations from what I considered noteworthy -Theres a lit...
nan (post): 3302 | More footage of the massive ships just above the surface of ...
PsychologicalRace739 (comment): 2836 | Steven Spielberg 

In [9]:
# Najczęstsze słowa w dyskusji
from collections import Counter
import re

words = Counter()
for text in df['content'].dropna():
    # Wyczyść tekst
    clean_text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words.update(clean_text.split())

# Usuń bardzo krótkie słowa
words = Counter({w: c for w, c in words.items() if len(w) > 3})

print("=== TOP 30 most common words ===")
for word, count in words.most_common(30):
    print(f"{word}: {count}")

# Wykres top words
top_30_words = dict(words.most_common(30))
fig = px.bar(x=list(top_30_words.values()), y=list(top_30_words.keys()),
             orientation='h', title='Top 30 Most Common Words',
             labels={'x': 'Frequency', 'y': 'Word'},
             color=list(top_30_words.values()),
             color_continuous_scale='Blues')
fig.update_layout(height=600, showlegend=False)
fig.show()

=== TOP 30 most common words ===
that: 16213
this: 12503
they: 9630
have: 7392
with: 7083
what: 5510
like: 5508
just: 5319
people: 4543
from: 4392
about: 4302
would: 4123
there: 3928
dont: 3832
some: 3315
think: 3276
more: 3149
your: 3102
know: 2979
them: 2902
their: 2826
aliens: 2708
these: 2691
were: 2550
will: 2543
because: 2433
even: 2318
when: 2298
been: 2264
other: 2177


In [10]:
# Długość treści (ile słów/znaków)
df['word_count'] = df['content'].fillna('').str.split().str.len()
df['char_count'] = df['content'].fillna('').str.len()

print("=== Content length statistics ===")
print(f"Avg words per content: {df['word_count'].mean():.0f}")
print(f"Avg chars per content: {df['char_count'].mean():.0f}")

print("\n=== Posts vs Comments length ===")
for activity in ['post', 'comment']:
    subset = df[df['activity_type'] == activity]
    print(f"\n{activity.upper()}:")
    print(f"  Avg words: {subset['word_count'].mean():.0f}")
    print(f"  Avg chars: {subset['char_count'].mean():.0f}")
    print(f"  Max words: {subset['word_count'].max()}")

# Wykresy długości treści
fig = make_subplots(rows=2, cols=2, subplot_titles=('Word Count Distribution', 'Char Count Distribution',
                                                     'Posts vs Comments - Words', 'Posts vs Comments - Chars'))

# Histogramy
fig.add_trace(go.Histogram(x=df['word_count'], nbinsx=30, name='Words', 
                           marker=dict(color='skyblue')), row=1, col=1)
fig.add_trace(go.Histogram(x=df['char_count'], nbinsx=30, name='Chars',
                           marker=dict(color='lightcoral')), row=1, col=2)

# Boxy plots
for activity_type in ['post', 'comment']:
    subset = df[df['activity_type'] == activity_type]
    fig.add_trace(go.Box(y=subset['word_count'], name=activity_type.capitalize(),
                         marker=dict(color='green' if activity_type == 'post' else 'orange')), row=2, col=1)
    fig.add_trace(go.Box(y=subset['char_count'], name=activity_type.capitalize(),
                         marker=dict(color='green' if activity_type == 'post' else 'orange')), row=2, col=2)

fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_yaxes(title_text="Frequency", row=1, col=2)
fig.update_yaxes(title_text="Word Count", row=2, col=1)
fig.update_yaxes(title_text="Char Count", row=2, col=2)
fig.update_layout(height=700, showlegend=False)
fig.show()

=== Content length statistics ===
Avg words per content: 32
Avg chars per content: 189

=== Posts vs Comments length ===

POST:
  Avg words: 131
  Avg chars: 831
  Max words: 2815

COMMENT:
  Avg words: 31
  Avg chars: 182
  Max words: 1476


In [11]:
# Analiza ról użytkowników na podstawie ich aktywności
user_stats = df.groupby('author').agg({
    'activity_id': 'count',  # total posts/comments
    'activity_type': lambda x: (x == 'post').sum(),  # number of posts
    'score': ['mean', 'sum', 'max'],  # engagement metrics
    'word_count': 'mean',  # avg content length
    'edited': lambda x: x.sum() if x.dtype == bool else (x == True).sum(),  # edits count
}).round(2)

user_stats.columns = ['total_activity', 'posts_count', 'avg_score', 'total_score', 'max_score', 'avg_word_count', 'edits_count']
user_stats = user_stats.sort_values('total_activity', ascending=False)

print("=== TOP USERS PROFILE ===")
print(user_stats.head(15))

# Wykresy profili użytkowników
top_15_users = user_stats.head(15)

fig = make_subplots(rows=2, cols=2, subplot_titles=('Total Activity', 'Avg Score per User',
                                                     'Posts Count', 'Avg Word Count'),
                     specs=[[{}, {}], [{}, {}]])

# Total activity
fig.add_trace(go.Bar(x=top_15_users['total_activity'].values, 
                     y=top_15_users.index, orientation='h',
                     marker=dict(color=top_15_users['total_activity'].values, colorscale='Viridis'),
                     name='Activity'), row=1, col=1)

# Avg score
fig.add_trace(go.Bar(x=top_15_users['avg_score'].values,
                     y=top_15_users.index, orientation='h',
                     marker=dict(color='lightgreen'),
                     name='Avg Score'), row=1, col=2)

# Posts count
fig.add_trace(go.Bar(x=top_15_users['posts_count'].values,
                     y=top_15_users.index, orientation='h',
                     marker=dict(color='orange'),
                     name='Posts'), row=2, col=1)

# Avg word count
fig.add_trace(go.Bar(x=top_15_users['avg_word_count'].values,
                     y=top_15_users.index, orientation='h',
                     marker=dict(color='lightcoral'),
                     name='Avg Words'), row=2, col=2)

fig.update_xaxes(title_text="Count", row=1, col=1)
fig.update_xaxes(title_text="Score", row=1, col=2)
fig.update_xaxes(title_text="Posts", row=2, col=1)
fig.update_xaxes(title_text="Words", row=2, col=2)
fig.update_layout(height=700, showlegend=False)
fig.show()

=== TOP USERS PROFILE ===
                      total_activity  posts_count  avg_score  total_score  \
author                                                                      
AutoModerator                    321            0       1.01          325   
aliens-ModTeam                   222            0       0.42           93   
lickem369                         91            1      -0.59          -54   
Funkadelick99                     75            1      -7.99         -599   
AntisocialGuru                    64            2       3.62          232   
MilkTeaPetty                      59            1      -0.25          -15   
Seekertwentyfifty                 52            1      12.50          650   
Mn4by                             45            0       6.96          313   
Pleasant-Lie-9053                 45            8      -0.38          -17   
Alien_Element                     45            1       4.31          194   
the_final_breath                  45            1 

In [12]:
# Klasyfikacja ról użytkowników (simple role classification)
# Rola opiera się na: aktywność, typ (posty vs komentarze), zaangażowanie (score)

def classify_user_role(author):
    # Handle NaN or empty author
    if pd.isna(author) or author == '':
        return 'deleted'
    
    # Check if author exists in user_stats
    if author not in user_stats.index:
        return 'unknown'
    
    user = user_stats.loc[author]
    
    # Post creators - głównie postują
    posts_ratio = user['posts_count'] / user['total_activity'] if user['total_activity'] > 0 else 0
    if posts_ratio > 0.5:
        role = 'Post Creator'
    # Active commenters - dużo komentarzy
    elif user['total_activity'] > user_stats['total_activity'].quantile(0.75):
        role = 'Active Commenter'
    # High engagement - wysokie wyniki
    elif user['avg_score'] > user_stats['avg_score'].quantile(0.75):
        role = 'Influencer'
    # Low activity
    elif user['total_activity'] <= 3:
        role = 'Lurker'
    else:
        role = 'Regular Participant'
    
    return role

# Przydziel role
df['user_role'] = df['author'].apply(classify_user_role)

print("=== USER ROLES DISTRIBUTION ===")
role_dist = df['user_role'].value_counts()
print(role_dist)

print("\n=== ROLES BY ACTIVITY TYPE ===")
role_activity = pd.crosstab(df['user_role'], df['activity_type'])
print(role_activity)

# Wykresy ról
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]],
                     subplot_titles=('Role Distribution', 'Roles by Activity Type'))

# Pie chart
fig.add_trace(go.Pie(labels=role_dist.index, values=role_dist.values, name='Roles'), row=1, col=1)

# Stacked bar chart
for role in role_activity.index:
    fig.add_trace(go.Bar(x=role_activity.columns, y=role_activity.loc[role],
                         name=role), row=1, col=2)

fig.update_xaxes(title_text="Activity Type", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(height=450)
fig.show()

=== USER ROLES DISTRIBUTION ===
user_role
Active Commenter    14519
Lurker              10440
deleted              5587
Influencer           3383
Post Creator          101
Name: count, dtype: int64

=== ROLES BY ACTIVITY TYPE ===
activity_type     comment  post
user_role                      
Active Commenter    14292   227
Influencer           3380     3
Lurker              10418    22
Post Creator           13    88
deleted              5546    41


## User Roles Explained

Każdy użytkownik jest klasyfikowany do jednej z następujących ról na podstawie jego profilu aktywności:

- **Post Creator**: Użytkownik głównie tworzy posty (>50% jego aktywności to posty, a nie komentarze). Inicjuje dyskusje i wprowadza nowe tematy.

- **Active Commenter**: Bardzo aktywny użytkownik (w top 25% pod względem liczby postów/komentarzy). Intensywnie uczestniczy w dyskusjach, głównie poprzez komentarze.

- **Influencer**: Użytkownik z wysokim średnim wynikiem (score), co sugeruje że jego treści są dobrze przyjmowane przez społeczność (top 25% po avg_score).

- **Lurker**: Użytkownik z bardzo niską aktywnością (≤3 posty/komentarze). Rzadko zabiera głos w dyskusjach.

- **Regular Participant**: Użytkownik o standardowej aktywności - nie pasuje do żadnej z powyższych kategorii. Uczestniczy w dyskusjach w umiarkowanym stopniu.

- **Deleted**: Autorzy, których konta zostały usunięte lub posty zostały skasowane.

In [13]:
# Analiza konwersacyjnych wątków - kto z kim rozmawia
# parent_id zawiera ID rodzica (postu lub komentarza, którego to dotyczy)

print("=== CONVERSATION STRUCTURE ===")

# Ile komentarzy każdy post dostał?
post_comments = df[df['activity_type'] == 'post'].copy()
post_replies = df[df['activity_type'] == 'comment'].copy()

print(f"Total posts: {len(post_comments)}")
print(f"Total comments: {len(post_replies)}")
print(f"Avg comments per post: {len(post_replies) / len(post_comments):.1f}")

# Komentarze na posty vs odpowiedzi na komentarze
direct_post_replies = post_replies[post_replies['parent_type'] == 'post']
comment_replies = post_replies[post_replies['parent_type'] == 'comment']

print(f"\nDirect replies to posts: {len(direct_post_replies)}")
print(f"Replies to comments: {len(comment_replies)}")
print(f"Nesting ratio: {len(comment_replies) / len(post_replies):.2%}")

# Wykresy struktury konwersacji
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]],
                     subplot_titles=('Posts vs Comments', 'Reply Type Distribution'))

# Pie chart - Posts vs Comments
fig.add_trace(go.Pie(labels=['Posts', 'Comments'],
                     values=[len(post_comments), len(post_replies)],
                     marker=dict(colors=['#FF6B6B', '#4ECDC4']),
                     name='Activity Type'), row=1, col=1)

# Bar chart - Reply types
reply_types = ['Direct Post Replies', 'Comment Replies']
reply_counts = [len(direct_post_replies), len(comment_replies)]
fig.add_trace(go.Bar(x=reply_types, y=reply_counts,
                     marker=dict(color=['#95E1D3', '#F38181']),
                     name='Reply Type'), row=1, col=2)

fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(height=400, showlegend=False)
fig.show()

=== CONVERSATION STRUCTURE ===
Total posts: 381
Total comments: 33649
Avg comments per post: 88.3

Direct replies to posts: 14613
Replies to comments: 19036
Nesting ratio: 56.57%


In [14]:
# Sentiment/Toxicity indicators (simple keyword-based)
toxic_keywords = ['hate', 'stupid', 'idiot', 'dumb', 'fake', 'conspiracy', 'lie', 'fake', 'hoax']
positive_keywords = ['good', 'great', 'excellent', 'love', 'thank', 'help', 'true', 'fact', 'evidence']

def count_keywords(text, keywords):
    if pd.isna(text):
        return 0
    text_lower = text.lower()
    return sum(1 for kw in keywords if kw in text_lower)

df['toxic_score'] = df['content'].apply(lambda x: count_keywords(x, toxic_keywords))
df['positive_score'] = df['content'].apply(lambda x: count_keywords(x, positive_keywords))
df['sentiment_ratio'] = (df['positive_score'] - df['toxic_score']) / (df['positive_score'] + df['toxic_score'] + 1)

print("=== TOXICITY ANALYSIS ===")
print(f"Posts with toxic language: {(df['toxic_score'] > 0).sum()} ({(df['toxic_score'] > 0).sum()/len(df):.1%})")
print(f"Posts with positive language: {(df['positive_score'] > 0).sum()} ({(df['positive_score'] > 0).sum()/len(df):.1%})")

print("\n=== TOXIC USERS (most toxic language) ===")
user_toxicity = df.groupby('author').agg({
    'toxic_score': 'sum',
    'activity_id': 'count'
}).rename(columns={'activity_id': 'posts_count'})
user_toxicity = user_toxicity[user_toxicity['posts_count'] > 2]  # filter for active users
user_toxicity['avg_toxic_per_post'] = user_toxicity['toxic_score'] / user_toxicity['posts_count']
user_toxicity = user_toxicity.sort_values('avg_toxic_per_post', ascending=False)
print(user_toxicity.head(10))

# Wykresy toksyczności
sentiment_counts = pd.Series({
    'With Toxic Language': (df['toxic_score'] > 0).sum(),
    'With Positive Language': (df['positive_score'] > 0).sum(),
    'Neutral': ((df['toxic_score'] == 0) & (df['positive_score'] == 0)).sum()
})

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]],
                     subplot_titles=('Sentiment Distribution', 'Top 10 Toxic Users'))

# Pie chart
fig.add_trace(go.Pie(labels=sentiment_counts.index, values=sentiment_counts.values,
                     marker=dict(colors=['#FF6B6B', '#4ECDC4', '#CCE5FF']),
                     name='Sentiment'), row=1, col=1)

# Top toxic users
top_toxic = user_toxicity.head(10)
fig.add_trace(go.Bar(x=top_toxic['avg_toxic_per_post'].values,
                     y=top_toxic.index, orientation='h',
                     marker=dict(color='#FF6B6B'),
                     name='Toxic Score'), row=1, col=2)

fig.update_xaxes(title_text="Avg Toxic Score per Post", row=1, col=2)
fig.update_layout(height=450, showlegend=False)
fig.show()

=== TOXICITY ANALYSIS ===
Posts with toxic language: 7930 (23.3%)
Posts with positive language: 5203 (15.3%)

=== TOXIC USERS (most toxic language) ===
                      toxic_score  posts_count  avg_toxic_per_post
author                                                            
SparkySpinz                     8            3            2.666667
Pure_Ad_9865                    9            4            2.250000
Gold-Engineering-543           11            5            2.200000
Opposite_Day_9771              24           11            2.181818
LudditeHorse                    6            3            2.000000
Prmarine110                     6            3            2.000000
dontcallmeray                   6            3            2.000000
Beansdacherry                   6            3            2.000000
melo1212                        6            3            2.000000
Lamarqe                         6            3            2.000000


## Score Explained

**Score** (wynik) to miara popularności i zaangażowania dla każdego postu lub komentarza na Reddit:

- **Średnia (Mean)** - przeciętny score dla wszystkich treści
- **Mediana (Median)** - środkowa wartość (50% wyżej, 50% niżej)
- **Maksimum** - najwyżej oceniany post/komentarz
- **Minimum** - najniżej oceniany post/komentarz

Score jest obliczany na podstawie głosów **upvote** (podoba mi się) i **downvote** (nie podoba mi się). Wysokie score oznacza że treść była dobrze przyjęta przez społeczność.

In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

# Przygotuj dane dla clustering
user_features = user_stats[['total_activity', 'posts_count', 'avg_score', 'avg_word_count', 'edits_count']].copy()
user_features = user_features.fillna(0)

# Normalizuj cechy
scaler = StandardScaler()
user_features_scaled = scaler.fit_transform(user_features)

# Elbow method - znajdź optymalny k
print("=== ELBOW METHOD ===")
inertias = []
silhouette_scores = []
K_range = range(1, 11)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(user_features_scaled)
    inertias.append(kmeans_temp.inertia_)
    print(f"k={k}: inertia={kmeans_temp.inertia_:.2f}")

# Elbow plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=inertias, 
                         mode='lines+markers',
                         name='Inertia',
                         line=dict(color='#FF6B6B', width=2),
                         marker=dict(size=8)))

fig.update_layout(
    title='Elbow Method For Optimal k',
    xaxis_title='Number of Clusters (k)',
    yaxis_title='Inertia (Sum of Squared Distances)',
    height=500,
    hovermode='x unified'
)
fig.show()

# Znajdź optymalny k (gdzie tempo spadku inercji się zmienia)
diffs = np.diff(inertias)
second_diffs = np.diff(diffs)
optimal_k = np.argmax(second_diffs) + 2  # +2 bo straciliśmy indeksy
print(f"\n📊 Sugerowany optymalny k: {optimal_k}")

=== ELBOW METHOD ===
k=1: inertia=70225.00
k=2: inertia=58375.40
k=3: inertia=50271.40
k=4: inertia=42133.73
k=5: inertia=35436.30
k=6: inertia=29492.61
k=7: inertia=25937.88
k=8: inertia=22018.93
k=9: inertia=19185.11
k=10: inertia=16266.53



📊 Sugerowany optymalny k: 2


In [19]:
# Clustering użytkowników metodą K-means na podstawie ich profili
# Użyj optymalnego k z elbow method

# Clustering
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
user_features['cluster'] = kmeans.fit_predict(user_features_scaled)

print(f"=== USER CLUSTERING (K-means, k={optimal_k}) ===")
cluster_dist = user_features['cluster'].value_counts().sort_index()
print(cluster_dist)

print("\n=== CLUSTER PROFILES ===")
cluster_profiles = []
for cluster in range(optimal_k):
    cluster_data = user_features[user_features['cluster'] == cluster]
    print(f"\nCluster {cluster} (n={len(cluster_data)}):")
    print(f"  Avg total activity: {cluster_data['total_activity'].mean():.1f}")
    print(f"  Avg posts: {cluster_data['posts_count'].mean():.1f}")
    print(f"  Avg score: {cluster_data['avg_score'].mean():.1f}")
    print(f"  Avg word count: {cluster_data['avg_word_count'].mean():.0f}")
    print(f"  Top members: {', '.join(cluster_data.nlargest(3, 'total_activity').index.tolist())}")
    
    cluster_profiles.append({
        'Cluster': f'Cluster {cluster}',
        'Size': len(cluster_data),
        'Avg Activity': cluster_data['total_activity'].mean(),
        'Avg Posts': cluster_data['posts_count'].mean(),
        'Avg Score': cluster_data['avg_score'].mean(),
        'Avg Words': cluster_data['avg_word_count'].mean()
    })

cluster_df = pd.DataFrame(cluster_profiles)

# Wykresy clustering
fig = make_subplots(rows=2, cols=2, 
                     subplot_titles=(f'Cluster Size Distribution (k={optimal_k})', 'Avg Total Activity',
                                    'Avg Score by Cluster', 'Avg Word Count'),
                     specs=[[{"type": "pie"}, {}], [{}, {}]])

# Pie chart - cluster sizes
fig.add_trace(go.Pie(labels=cluster_df['Cluster'], values=cluster_df['Size'],
                     name='Cluster Size'), row=1, col=1)

# Bar charts
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3', '#F38181', '#FFB6C1'][:optimal_k]

fig.add_trace(go.Bar(x=cluster_df['Cluster'], y=cluster_df['Avg Activity'],
                     marker=dict(color=colors[0] if optimal_k > 0 else '#FF6B6B'),
                     name='Avg Activity'), row=1, col=2)

fig.add_trace(go.Bar(x=cluster_df['Cluster'], y=cluster_df['Avg Score'],
                     marker=dict(color=colors[1] if optimal_k > 1 else '#4ECDC4'),
                     name='Avg Score'), row=2, col=1)

fig.add_trace(go.Bar(x=cluster_df['Cluster'], y=cluster_df['Avg Words'],
                     marker=dict(color=colors[2] if optimal_k > 2 else '#95E1D3'),
                     name='Avg Words'), row=2, col=2)

fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Score", row=2, col=1)
fig.update_yaxes(title_text="Words", row=2, col=2)
fig.update_layout(height=700, showlegend=False)
fig.show()

=== USER CLUSTERING (K-means, k=6) ===
cluster
0    12618
1      630
2      521
3        2
4      261
5       13
Name: count, dtype: int64

=== CLUSTER PROFILES ===

Cluster 0 (n=12618):
  Avg total activity: 1.7
  Avg posts: 0.0
  Avg score: 8.6
  Avg word count: 19
  Top members: Acrobatic-Focus-3547, ChapterSpecial6920, shadowmage666

Cluster 1 (n=630):
  Avg total activity: 1.9
  Avg posts: 0.0
  Avg score: 7.5
  Avg word count: 152
  Top members: KetamineSNORTER1, Krystami, markthedeadmet

Cluster 2 (n=521):
  Avg total activity: 5.9
  Avg posts: 0.0
  Avg score: 14.1
  Avg word count: 54
  Top members: Mn4by, Alien_Element, The_Scout1255

Cluster 3 (n=2):
  Avg total activity: 271.5
  Avg posts: 0.0
  Avg score: 0.7
  Avg word count: 70
  Top members: AutoModerator, aliens-ModTeam

Cluster 4 (n=261):
  Avg total activity: 9.4
  Avg posts: 1.2
  Avg score: 18.0
  Avg word count: 44
  Top members: lickem369, Funkadelick99, AntisocialGuru

Cluster 5 (n=13):
  Avg total activity: 2.5

In [ ]:
## Wnioski z Klastrowania (K=6)

### 📊 **Cluster 0 - "Silent Lurkers" (12,618 osób - 97%!)**
- **Charakterystyka**: Prawie wszyscy użytkownicy to osoby z minimalną aktywnością (1.7 post/komentarz średnio)
- **Score**: 8.6 - niskie, co sugeruje że ich rzadkie wypowiedzi są słabo oceniane
- **Treść**: Bardzo krótkie komentarze (19 słów)
- **Wniosek**: Zdecydowana większość to "lurkers" - czytają ale prawie nie piszą. Dyskusja jest napędzana przez mniejszość.

### 📝 **Cluster 1 - "Essay Writers" (630 osób)**
- **Charakterystyka**: Osoby piszące długie, szczegółowe komentarze (152 słowa!)
- **Score**: 7.5 - średnie
- **Wniosek**: Mogą to być ludzie piszący "walls of text" - mogą być wartościowi ale mogą też być zignorkowani z powodu długości

### 🎯 **Cluster 2 - "Active Commenters" (521 osób)**
- **Charakterystyka**: Umiarkowanie aktywni (5.9 post/komentarz), piszą sensownie (54 słowa)
- **Score**: 14.1 - wysokie, jeden z najwyższych!
- **Wniosek**: To "solidni uczestnicy" dyskusji - piszą sensowne komentarze średniej długości, które są dobrze oceniane

### 🤖 **Cluster 3 - "Moderators/Bots" (2 osoby)**
- **Charakterystyka**: AutoModerator i moderatorzy - super aktywni (271.5!)
- **Score**: 0.7 - bardzo niskie (bo to automaty)
- **Wniosek**: Moderacja dyskusji, są wszędzie ale ich głos nie jest oceniany jak zwykłych użytkowników

### ⭐ **Cluster 4 - "Content Creators" (261 osób)**
- **Charakterystyka**: Najaktywniejsi użytkownicy (9.4 post/komentarz), tworzą zarówno posty (1.2) jak i komentarze
- **Score**: 18.0 - NAJWYŻSZY! Ich treści są wysoce cenione
- **Treść**: Zwięzłe (44 słowa) ale efektywne
- **Wniosek**: To są **liderzy dyskusji** - tworzą wartościową treść (posty + komentarze) i są dobrze odbierani

### 🏆 **Cluster 5 - "Influencers" (13 osób)**
- **Charakterystyka**: Piszą rzadko (2.5 średnio) ale kiedy piszą... BOOM!
- **Score**: 1813.0 - astronomiczny! Prawie 100 wyższy niż pozostali
- **Wniosek**: Super-ludzie. Każdy ich komentarz zbiera tysiące upvotów. To są kingmakers w dyskusji.

---

## 🎯 Główne Obserwacje

1. **Piramida zaangażowania**: Dyskusja jest skoncentrowana na BARDZO małej grupie (Cluster 4+5 = ~274 osób z ~14k)
2. **Toksyczność mogą napędzać**: Influencerzy (Cluster 5) i Content Creators (Cluster 4) - im się słucha
3. **Długość nie zawsze pomaga**: Essay Writers (Cluster 1) piszą więcej ale mają niższy score niż Active Commenters
4. **Moderacja jest widoczna**: AutoModerator ma ~272 akcji - to spora aktywność moderacyjna
5. **Polaryzacja treści**: Rozpasanie to głównie coś dla "elity" (Cluster 5), zwykli ludzie to tylko czytają

SyntaxError: invalid decimal literal (987405401.py, line 32)